<a href="https://colab.research.google.com/github/patriciarrs/RAG/blob/main/PatriciaSilva_RAG_FinalProject.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 1. Final Project Information - RAG [don't edit this section]

## IMPORTANT
**Add comments to your code whenever necessary to explain your logic.**

## Submission
- Single Deliverable: Google Colab Notebook
- File Name Format: [YourName]_RAG_FinalProject.ipynb (optional but it helps us track your file)
- Submit as: ZIP file containing only the .ipynb file (the student portal doesn't accpet .ipynb files)

## Mandatory
- Ingestion Phase
- Inference Phase

# 2. Project Specific Information

Add all the information you find useful to explain your project, the steps you took, the features implemented, and possible enhancements for the future.

## Project Description
[2-3 sentences explaining what your RAG system does and what documents it works with]

## Steps
1. Data collection and preprocessing
2. ...

## Features Implemented
- [x] RAG v2 baseline (mandatory)
- [ ] < additional feature >

# 3. Installation

In [1]:
%pip install "langchain==0.3.27" -qqq
%pip install "langchain-community==0.3.31" -qqq
%pip install "langchain-openai==0.3.35" -qqq
%pip install "langchain-chroma==0.2.6" -qqq
%pip install pypdf -qqq
%pip install flashrank -qqq
%pip install ragas -qqq

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 19.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 461.3/461.3 kB 22.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.5/66.5 kB 4.3 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
langgraph 1.1.9 requires langchain-core<2,>=1.3.0, but you have langchain-core 0.3.86 which is incompatible.
langgraph-prebuilt 1.0.10 requires langchain-core>=1.0.0, but you have langchain-core 0.3.86 which is incompatible.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 35.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 3.0 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the so

# 4. Setup

## Load Secrets Function

In [2]:
def load_secrets(keys):
    import os

    try:
        from google.colab import userdata  # type: ignore
        values = {key: userdata.get(key) for key in keys}
        in_colab = True
    except Exception:
        # Load from .env file for local development
        try:
            from dotenv import load_dotenv
            load_dotenv()
        except ImportError:
            raise ImportError(
                "python-dotenv is required for local development. "
                "Install it with: pip install python-dotenv"
            )

        # Get values from environment (loaded from .env file)
        values = {key: os.getenv(key) for key in keys}
        in_colab = False

    missing = [key for key, value in values.items() if not value]
    if missing:
        if in_colab:
            raise ValueError(f"Missing keys: {', '.join(missing)}. Set them in Colab Secrets.")
        else:
            env_file_help = (
                f"Missing keys: {', '.join(missing)}.\n\n"
                "Create a .env file in the project root with:\n"
                + "\n".join([f"{key}=YOUR_VALUE_HERE" for key in missing])
            )
            raise ValueError(env_file_help)

    for key, value in values.items():
        os.environ[key] = value

    return values

In [4]:
try:
    secrets = load_secrets(['PINECONE_API_KEY', 'PINECONE_TENANT', 'PINECONE_DATABASE', 'OPENAI_API_KEY'])
except ValueError as e:
    print(e)

# 5. Global Variables

In [6]:
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_chroma import Chroma
from langchain.retrievers import ParentDocumentRetriever
from langchain.storage import InMemoryStore
from langchain.text_splitter import RecursiveCharacterTextSplitter

VERSION = "evaluation"

COLLECTION_NAME = f"wiki_docs_{VERSION}"

# Embeddings Model
embeddings = OpenAIEmbeddings(
    model="text-embedding-3-small"
)

# LLM
llm = ChatOpenAI(
    model="gpt-4o-mini"
)

# Classification LLM
classification_llm = ChatOpenAI(
    model="gpt-3.5-turbo",
    temperature=0
)

#--------------------------------------------------------------------------------
# TWO VECTORSTORES FOR EVALUATION
#--------------------------------------------------------------------------------
# Main vectorstore (baseline + multi-query strategies)
vectorstore = Chroma(
  embedding_function=embeddings,
  collection_name=COLLECTION_NAME,
  chroma_cloud_api_key=secrets["CHROMA_API_KEY"],
  tenant=secrets["CHROMA_TENANT"],
  database=secrets["CHROMA_DATABASE"]
)

# Vectorstore for child chunks (parent-child strategies)
vectorstore_children = Chroma(
  embedding_function=embeddings,
  collection_name=f"{COLLECTION_NAME}_parent_children",
  chroma_cloud_api_key=secrets["CHROMA_API_KEY"],
  tenant=secrets["CHROMA_TENANT"],
  database=secrets["CHROMA_DATABASE"]
)

# Parent splitter (large chunks for context)
parent_splitter = RecursiveCharacterTextSplitter(
    chunk_size=2000,   # Large chunks for rich context in answers
    chunk_overlap=400, # 20% overlap
)

# Child splitter (small chunks for search)
child_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,   # Small chunks for precise search matching
    chunk_overlap=75, # 15% overlap
)

# InMemoryStore for parent documents
parent_docstore = InMemoryStore()

# ParentDocumentRetriever
parent_retriever = ParentDocumentRetriever(
    vectorstore=vectorstore_children,
    docstore=parent_docstore,
    child_splitter=child_splitter,
    parent_splitter=parent_splitter
)

KeyError: 'CHROMA_API_KEY'

# 6. Indexing / Ingestion Phase

In [ ]:
# =========================================
# Example Steps:
# =========================================
# Load documents (data)
# Chunk documents
# Create embeddings
# Store in vector database

# 7. Inference Phase (RAG)

In [ ]:
# =========================================
# Example Steps:
# =========================================
# Retriever setup (if not declared during setup)
# Prompt template
# LLM integration
# LCEL chain creation

# 8. User Interface (Gradio or other) - optional

In [ ]:
# =========================================
# Example Steps:
# =========================================
# Gradio UI implementation
# Query processing function
# Launch interface

# 9. Testing / Demo

In [ ]:
# =========================================
# Example Steps:
# =========================================
# Example Queries to Test
# Functions to Test
# Demo information